In [ ]:
# --- INSTALLATION (Run this once if not already installed) ---
!pip install -q -U transformers accelerate timm torch datasets
!pip install -q sacremoses sentencepiece # mBART dependencies

# --- Core Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, UnidentifiedImageError
from transformers import (
    MBart50TokenizerFast,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)
from transformers.modeling_outputs import BaseModelOutput
import timm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import os
import json
import pandas as pd
import random
from transformers.modeling_outputs import BaseModelOutput
from contextlib import nullcontext

In [ ]:
# --- Device Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Prefer TF32 on Ampere+; harmless elsewhere
import torch
torch.backends.cuda.matmul.allow_tf32 = True                
torch.backends.cudnn.allow_tf32 = True  

# --- Global Paths (Adjust if your Kaggle paths are different) ---
# DATA_DIR = "/kaggle/input/synthetic-bic-pairs/synthetic_bengali_images"
DATA_DIR = "/kaggle/input/syn-images/syn-ben-images"
REAL_IMAGE_BASE_DIR = "/kaggle/input/coco-image-caption/train2014/train2014"
COCO_ANNOTATIONS_PATH = '/kaggle/input/coco-image-caption/annotations_trainval2014/annotations/captions_train2014.json'
MODEL_SAVE_PATH = "/kaggle/working/maxvit_mbart_captioning_model.pth"
MODEL_INPUT_PATH = "/kaggle/input/train/transformers/default/12/maxvit_mbart_captioning_model.pth"
MODEL_CHECKPOINT_PATH = "/kaggle/working/maxvit_mbart_bn_checkpoint.pt"
MODEL_CHECKPOINT_PATH_IN = "/kaggle/input/train/transformers/default/16/maxvit_mbart_bn_checkpoint.pt"

# --- Model and Tokenizer Names ---
MAXVIT_MODEL_NAME = "maxvit_base_tf_224.in1k"
TOKENIZER_MODEL_NAME = "google/mt5-base"
DECODER_MODEL_NAME = "google/mt5-base"
MBART_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt" 

# --- Training Configuration (Adjust as needed) ---
BATCH_SIZE = 2                                               
GRAD_ACCUM_STEPS = 4                                        
LEARNING_RATE = 1e-4                                       
NUM_EPOCHS = 6
PATCH_ALIGNMENT_LOSS_WEIGHT = 0.5
MAX_CAPTION_LENGTH = 96                                   
INITIAL_PATCH_ALIGNMENT_WEIGHT = 0.5

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.ToTensor(),
    # Use ImageNet mean and std for normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# --- Utility Functions for Data Loading ---

def extract_captions(full_caption):
    if "In Bengali:" in full_caption:
        caption_en = full_caption.split("In Bengali:")[0].strip()
        caption_bn = full_caption.split("In Bengali:")[-1].strip()
    else:
        parts = full_caption.strip().split(". ")
        if len(parts) >= 2:
            caption_en = parts[0].strip()
            caption_bn = parts[-1].strip()
        else:
            caption_en = full_caption.strip()
            caption_bn = full_caption.strip()
    if caption_en.startswith("A photo of: "):
        caption_en = caption_en[len("A photo of: "):].strip()
    return caption_en, caption_bn

def handle_full_stop_variation(caption_en):
    captions = [caption_en]
    if caption_en.endswith('.'):
        captions.append(caption_en[:-1].strip())
    if not caption_en.endswith('.'):
        captions.append(caption_en + '.')
    return captions

def safe_load_image(img_path: str):
    try:
        img = Image.open(img_path).convert('RGB')
        return img
    except FileNotFoundError:
        print(f"ERROR: Image file not found at: {img_path}")
        return None
    except UnidentifiedImageError:
        print(f"ERROR: Cannot identify image file (corrupted or unsupported format): {img_path}")
        return None
    except Exception as e:
        print(f"ERROR: An unexpected error occurred while loading image {img_path}: {e}")
 
        return None

import unicodedata

import unicodedata

def clean_bengali_caption(caption_text):
    import unicodedata
    cleaned = unicodedata.normalize('NFKC', str(caption_text))
    cleaned = (cleaned.replace('‘', "'").replace('’', "'")
                     .replace('“', '"').replace('”', '"')
                     .replace('—', '-').replace('–', '-')
                     .replace('…', '...'))
    cleaned = ''.join(ch for ch in cleaned if ch.isprintable())
    cleaned = ' '.join(cleaned.split()).strip()
    # strip any lone surrogates / odd bytes that can upset the fast tokenizer
    cleaned = cleaned.encode('utf-8', 'ignore').decode('utf-8', 'ignore')   # <<< added
    return cleaned

# --- Custom Dataset Class (Using torchvision transforms) ---
class BengaliCaptionDataset(Dataset):
    def __init__(self, df_results, image_transform, tokenizer, max_length=MAX_CAPTION_LENGTH):
        self.df = df_results
        self.image_transform = image_transform
        self.tokenizer = tokenizer
        self.max_length = int(max_length)
        self.pad_id = int(self.tokenizer.pad_token_id)  
        
        self.tokenizer.src_lang = "bn_IN"
        self.tokenizer.tgt_lang = "bn_IN"

        print(f"Dataset initialized with {len(self.df)} entries.")
        
        self.successfully_tokenized_count = 0
        self.failed_tokenization_count = 0



    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        real_image_path = row['real_image_path']
        generated_image_path = row['generated_image_path']
        caption_bn = row["caption_bn"]
        caption_bn = clean_bengali_caption(caption_bn)

        real_img = safe_load_image(real_image_path)
        generated_img = safe_load_image(generated_image_path)

        if real_img is None or generated_img is None:
            return None

        try:

            if not isinstance(real_img, Image.Image):
                return None # Ensure it's skipped
            if not isinstance(generated_img, Image.Image):
                return None # Ensure it's skipped
            
            real_pixel_values = self.image_transform(real_img)
           
            synthetic_pixel_values = self.image_transform(generated_img)
           
        except Exception as e:
            return None

      
        try:
        
            try:
                tokens = self.tokenizer(
                    caption_bn,
                    return_tensors="pt",
                    truncation=True,
                    max_length=int(self.max_length),                 # <<< ensure int
                    padding="max_length"
                )
            except OverflowError:
                # fallback to slow tokenizer just for this sample
                if not hasattr(self, "_tok_slow"):
                    self._tok_slow = MBart50TokenizerFast.from_pretrained(
                        "facebook/mbart-large-50-many-to-many-mmt",
                        src_lang="bn_IN", tgt_lang="bn_IN"
                    )
                tokens = self._tok_slow(
                    caption_bn,
                    return_tensors="pt",
                    truncation=True,
                    max_length=int(self.max_length),
                    padding="max_length"
                )
            except Exception as e:
                print(f"Tokenizer failed for idx {idx}: {e}")
                return None

            labels = tokens.input_ids.squeeze(0)
            attention_mask = tokens["attention_mask"].squeeze(0)
            
            # mask PAD tokens in labels so CE ignores them
            pad_id = self.tokenizer.pad_token_id
            labels[labels == pad_id] = -100
            
            # guard: if everything is ignored, skip sample to avoid CE NaN
            if torch.all(labels == -100):
                return None

            # try:
            #     debug_text = self.tokenizer.batch_decode([labels.tolist()], skip_special_tokens=True)[0]
            # except Exception as e:
            #     # Don't fail the sample on decode bugs
            #     debug_text = None

            
            self.successfully_tokenized_count += 1

        except Exception as e:
            print(f"❌❌❌ CRITICAL ERROR during tokenization for index {idx} ❌❌❌")
            print(f"  Problematic caption: '{caption_bn}'")
            print(f"  Type of problematic caption: {type(caption_bn)}")
            print(f"  Length of problematic caption: {len(caption_bn)}")
            print(f"  FULL EXCEPTION: {type(e).__name__}: {e}")
            self.failed_tokenization_count += 1
            return None

        return {
            "real_pixel_values": real_pixel_values,
            "synthetic_pixel_values": synthetic_pixel_values,
            "labels": labels,
            "attention_mask": attention_mask
        }

# --- Custom Collate Function for DataLoader ---
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return None

    real_pixel_values = torch.stack([item["real_pixel_values"] for item in batch])
    synthetic_pixel_values = torch.stack([item["synthetic_pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])

    return {
        "real_pixel_values": real_pixel_values,
        "synthetic_pixel_values": synthetic_pixel_values,
        "labels": labels,
        "attention_mask": attention_mask
    }



In [ ]:
from collections import Counter
import traceback
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # cope with truncated JPEGs
from transformers.modeling_outputs import BaseModelOutput  # if not already imported


# --- Debug wrapper to pinpoint failing transform op ---
class DebugCompose(torch.nn.Module):
    def __init__(self, ops):
        super().__init__()
        self.ops = ops
    def forward(self, img, path_hint=""):
        x = img
        for i, op in enumerate(self.ops):
            try:
                x = op(x)
            except Exception as e:
                print(f"\n[TRANSFORM FAIL] file={path_hint}\n  op#{i}: {op}\n  error: {type(e).__name__}: {e}")
                traceback.print_exc(limit=1)
                raise
        return x

# --- Safer RandomResizedCrop with fallback so the sample is never dropped ---
class SafeRandomResizedCrop(torch.nn.Module):
    def __init__(self, size, scale=(0.5, 1.0), ratio=(0.75, 1.3333)):
        super().__init__()
        self.rrc = transforms.RandomResizedCrop(size, scale=scale, ratio=ratio)
        self.fallback = transforms.Compose([transforms.Resize((size, size)), transforms.CenterCrop(size)])
    def forward(self, img):
        try:
            return self.rrc(img)
        except Exception:
            return self.fallback(img)

# --- (A) DEBUG transform (only for preflight) ---
debug_ops = [
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),  # original – we want to see if this fails
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]
debug_transform = DebugCompose(debug_ops)

# --- (B) SAFE training transform (use for actual training) ---
train_transform_safe = transforms.Compose([
    transforms.Resize((256, 256)),
    SafeRandomResizedCrop(224, scale=(0.5, 1.0)),  # more forgiving + fallback
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.0),  # disable; can hurt caption semantics
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



def preflight_filter(df_results, tokenizer, image_transform, max_length, use_debug=False):
    """
    Validates each row once, logs reasons, and returns indices to keep.
    If use_debug=True, prints the exact op/file that failed.
    """
    reasons = Counter()
    keep = []

    for i, row in enumerate(tqdm(df_results.itertuples(index=True), total=len(df_results), desc="Preflight")):
        # 1) load images
        real_img = safe_load_image(row.real_image_path)
        if real_img is None:
            reasons["real_image_load_fail"] += 1; continue
        gen_img  = safe_load_image(row.generated_image_path)
        if gen_img is None:
            reasons["gen_image_load_fail"] += 1; continue

        # 2) transforms
        try:
            if use_debug:
                _ = image_transform(real_img, row.real_image_path)
                _ = image_transform(gen_img,  row.generated_image_path)
            else:
                _ = image_transform(real_img)
                _ = image_transform(gen_img)
        except Exception:
            reasons["transform_fail"] += 1; continue

        # 3) tokenization
        text = clean_bengali_caption(row.caption_bn)
        try:
            toks = tokenizer(text, return_tensors="pt", truncation=True,
                             max_length=int(max_length), padding="max_length")
        except Exception:
            reasons["tokenize_fail"] += 1; continue

        input_ids = toks["input_ids"].squeeze(0)
        attn_mask = toks["attention_mask"].squeeze(0)
        pad_id = int(tokenizer.pad_token_id)

        labels = input_ids.clone()
        labels[labels == pad_id] = -100

        if attn_mask.sum().item() == 0:
            reasons["all_pad_attn_mask"] += 1; continue
        if torch.all(labels == -100):
            reasons["all_ignore_labels"] += 1; continue

        keep.append(row.Index)

    print("\nPreflight summary:", dict(reasons))
    return keep

In [ ]:
from collections import Counter
import traceback
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # cope with truncated JPEGs
from transformers.modeling_outputs import BaseModelOutput  # if not already imported


# --- Debug wrapper to pinpoint failing transform op ---
class DebugCompose(torch.nn.Module):
    def __init__(self, ops):
        super().__init__()
        self.ops = ops
    def forward(self, img, path_hint=""):
        x = img
        for i, op in enumerate(self.ops):
            try:
                x = op(x)
            except Exception as e:
                print(f"\n[TRANSFORM FAIL] file={path_hint}\n  op#{i}: {op}\n  error: {type(e).__name__}: {e}")
                traceback.print_exc(limit=1)
                raise
        return x

# --- Safer RandomResizedCrop with fallback so the sample is never dropped ---
class SafeRandomResizedCrop(torch.nn.Module):
    def __init__(self, size, scale=(0.5, 1.0), ratio=(0.75, 1.3333)):
        super().__init__()
        self.rrc = transforms.RandomResizedCrop(size, scale=scale, ratio=ratio)
        self.fallback = transforms.Compose([transforms.Resize((size, size)), transforms.CenterCrop(size)])
    def forward(self, img):
        try:
            return self.rrc(img)
        except Exception:
            return self.fallback(img)

# --- (A) DEBUG transform (only for preflight) ---
debug_ops = [
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),  # original – we want to see if this fails
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]
debug_transform = DebugCompose(debug_ops)

# --- (B) SAFE training transform (use for actual training) ---
train_transform_safe = transforms.Compose([
    transforms.Resize((256, 256)),
    SafeRandomResizedCrop(224, scale=(0.5, 1.0)),  # more forgiving + fallback
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.0),  # disable; can hurt caption semantics
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



def preflight_filter(df_results, tokenizer, image_transform, max_length, use_debug=False):
    """
    Validates each row once, logs reasons, and returns indices to keep.
    If use_debug=True, prints the exact op/file that failed.
    """
    reasons = Counter()
    keep = []

    for i, row in enumerate(tqdm(df_results.itertuples(index=True), total=len(df_results), desc="Preflight")):
        # 1) load images
        real_img = safe_load_image(row.real_image_path)
        if real_img is None:
            reasons["real_image_load_fail"] += 1; continue
        gen_img  = safe_load_image(row.generated_image_path)
        if gen_img is None:
            reasons["gen_image_load_fail"] += 1; continue

        # 2) transforms
        try:
            if use_debug:
                _ = image_transform(real_img, row.real_image_path)
                _ = image_transform(gen_img,  row.generated_image_path)
            else:
                _ = image_transform(real_img)
                _ = image_transform(gen_img)
        except Exception:
            reasons["transform_fail"] += 1; continue

        # 3) tokenization
        text = clean_bengali_caption(row.caption_bn)
        try:
            toks = tokenizer(text, return_tensors="pt", truncation=True,
                             max_length=int(max_length), padding="max_length")
        except Exception:
            reasons["tokenize_fail"] += 1; continue

        input_ids = toks["input_ids"].squeeze(0)
        attn_mask = toks["attention_mask"].squeeze(0)
        pad_id = int(tokenizer.pad_token_id)

        labels = input_ids.clone()
        labels[labels == pad_id] = -100

        if attn_mask.sum().item() == 0:
            reasons["all_pad_attn_mask"] += 1; continue
        if torch.all(labels == -100):
            reasons["all_ignore_labels"] += 1; continue

        keep.append(row.Index)

    print("\nPreflight summary:", dict(reasons))
    return keep

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from transformers import AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput


def _l2n(x, dim=-1, eps=1e-8):
    """L2-normalize along dim with numerical guard."""
    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))


class MaxVitMbartCaptioningModel(nn.Module):
    def __init__(
        self,
        encoder_model_name,
        decoder_model_name,
        bn_in_token_id,
        patch_alignment_weight=0.05,

        # == A: stability knobs ==
        attn_temp: float = 1.0,            # temperature for cross-attn -> softmax
        topk_ratio: float = 0.30,          # keep top-k% patches when forming weights
        use_last_k_cross_layers: int = 2,  # use only final K decoder layers’ cross-attn

        # efficiency
        pool_factor: int = 1,              # 2 or 3 reduces HxW in PAL/OT

        # == B: InfoNCE (optional) ==
        use_info_nce: bool = False,
        info_nce_temp: float = 0.07,
        info_nce_coef: float = 0.3,        # scaled by patch_alignment_weight

        # == C: OT/Sinkhorn (optional) ==
        use_ot: bool = False,
        ot_reg: float = 0.05,              # entropic regularization (epsilon)
        ot_iters: int = 50,
        ot_topk_ratio: float = 0.15,       # sparsify patches before OT
        ot_coef: float = 0.5,              # scaled by patch_alignment_weight

        # == E: multi-scale PAL (optional) ==
        multi_scale: bool = False
    ):
        super().__init__()

        # ---------------------------
        # Vision encoder (frozen)
        # ---------------------------
        self.vision_encoder = timm.create_model(
            encoder_model_name, pretrained=True, features_only=True
        )
        self.vision_encoder.eval()
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        if hasattr(self.vision_encoder, "set_grad_checkpointing"):
            self.vision_encoder.set_grad_checkpointing(True)

        ch_list = self.vision_encoder.feature_info.channels()
        self.last_ch = ch_list[-1]
        self.prev_ch = ch_list[-2] if len(ch_list) >= 2 else None

        # ------------------------------------------
        # Language decoder (defines the "canonical"
        # dtype we keep for projections/LayerNorm)
        # ------------------------------------------
        self.language_decoder = AutoModelForSeq2SeqLM.from_pretrained(
            decoder_model_name, attn_implementation="eager", low_cpu_mem_usage=True
        )
        self.language_decoder.config.decoder_start_token_id = bn_in_token_id
        self.language_decoder.config.use_cache = False
        if hasattr(self.language_decoder, "gradient_checkpointing_enable"):
            self.language_decoder.gradient_checkpointing_enable()

        self.d_model = self.language_decoder.config.d_model
        # decoder param dtype (usually float32 for mBART)
        self.dec_dtype = next(self.language_decoder.parameters()).dtype

        # Projections + LN must match decoder dtype to avoid Half/Float mismatches under AMP
        self.vision_projection = nn.Linear(self.last_ch, self.d_model).to(self.dec_dtype)
        self.encoder_norm = nn.LayerNorm(self.d_model, eps=1e-5).to(self.dec_dtype)
        nn.init.xavier_uniform_(self.vision_projection.weight)
        if self.vision_projection.bias is not None:
            nn.init.zeros_(self.vision_projection.bias)

        # Optional multi-scale projection for the penultimate feature map
        self.multi_scale = bool(multi_scale and self.prev_ch is not None)
        if self.multi_scale:
            self.vision_projection_prev = nn.Linear(self.prev_ch, self.d_model).to(self.dec_dtype)
            nn.init.xavier_uniform_(self.vision_projection_prev.weight)
            if self.vision_projection_prev.bias is not None:
                nn.init.zeros_(self.vision_projection_prev.bias)

        # knobs
        self.patch_alignment_weight = float(patch_alignment_weight)
        self.attn_temp = float(attn_temp)
        self.topk_ratio = float(topk_ratio)
        self.use_last_k_cross_layers = int(max(1, use_last_k_cross_layers))
        self.pool_factor = int(max(1, pool_factor))

        # InfoNCE
        self.use_info_nce = bool(use_info_nce)
        self.info_nce_temp = float(info_nce_temp)
        self.info_nce_coef = float(info_nce_coef)

        # OT
        self.use_ot = bool(use_ot)
        self.ot_reg = float(ot_reg)
        self.ot_iters = int(ot_iters)
        self.ot_topk_ratio = float(ot_topk_ratio)
        self.ot_coef = float(ot_coef)

    # =========================
    # Encoding helpers
    # =========================
    def _encode_feats_list(self, pixel_values):
        """Return [prev?, last] feature maps; optionally downsampled by pool_factor."""
        with torch.no_grad():
            feats_list = self.vision_encoder(pixel_values)     # list of feature maps
        outs = []

        idxs = [-1] if not self.multi_scale else [-2, -1]
        for i in idxs:
            f = feats_list[i]                                  # (B,C,H,W)
            if self.pool_factor > 1:
                H, W = f.shape[-2:]
                f = F.adaptive_avg_pool2d(
                    f, (max(1, H // self.pool_factor), max(1, W // self.pool_factor))
                )
            outs.append(f)
        return outs  # [prev?, last]

    def _project_norm(self, fmap, use_prev: bool = False):
        """Project (B,C,H,W) -> (B,S,D) with dtype aligned to projection weights + LN dtype."""
        B, C, H, W = fmap.shape
        S = H * W
        patches = fmap.reshape(B, C, S).transpose(1, 2)        # (B,S,C)

        # Choose the right projection head
        proj = self.vision_projection_prev if use_prev else self.vision_projection

        # *** CRITICAL: cast inputs to the Linear's dtype to avoid Half/Float mismatch ***
        patches = patches.to(dtype=proj.weight.dtype)

        enc = proj(patches)                                    # (B,S,D)
        # Keep LN dtype consistent with its parameters (same as decoder dtype)
        enc = self.encoder_norm(enc.to(self.encoder_norm.weight.dtype))
        return enc, (H, W)

    # =========================
    # Cross-attn -> patch weights
    # =========================
    def _attention_to_patch_weights(self, cross_attentions, attention_mask, S):
        """Average last-K cross-attn maps -> (B,S) weights with temp + top-k sparsity."""
        keep = cross_attentions[-self.use_last_k_cross_layers:]
        attn = [torch.nan_to_num(a.float(), 0.0, 0.0, 0.0) for a in keep if a is not None]
        if not attn:
            B = attention_mask.size(0) if attention_mask is not None else 1
            device = (
                attention_mask.device
                if attention_mask is not None
                else next(self.language_decoder.parameters()).device
            )
            return torch.full((B, S), 1.0 / S, device=device)

        cross = torch.stack(attn, dim=0)  # (L,B,H,T,S)

        # Weighted average over T using attention_mask
        if attention_mask is not None:
            T = cross.size(3)
            tgt = attention_mask[:, :T].to(cross.dtype).view(1, -1, 1, T, 1)  # (1,B,1,T,1)
            cross = cross * tgt
            denom = tgt.sum(dim=3, keepdim=True).clamp_min(1.0)
            cross = cross.sum(dim=3, keepdim=True) / denom
            cross = cross.squeeze(3)                      # (L,B,H,S)
        else:
            cross = cross.mean(dim=3)

        # avg heads -> (L,B,S), then layers -> (B,S)
        w = cross.mean(dim=2).mean(dim=0)                 # (B,S)

        # temperature + softmax
        w = torch.softmax(w / max(1e-6, self.attn_temp), dim=-1)

        # sparsify (top-k) to reduce background noise
        if 0.0 < self.topk_ratio < 1.0:
            k = max(1, min(int(self.topk_ratio * w.size(1)), w.size(1)))
            topv, topi = torch.topk(w, k, dim=1)
            mask = torch.zeros_like(w).scatter_(1, topi, 1.0)
            w = w * mask
            w = w / w.sum(dim=1, keepdim=True).clamp_min(1e-8)

        bad = ~torch.isfinite(w).all(dim=1, keepdim=True)
        if bad.any():
            uniform = torch.full_like(w, 1.0 / w.size(1))
            w = torch.where(bad, uniform, w)
        return w  # (B,S)

    # =========================
    # Sinkhorn OT (per-sample)
    # =========================
    def _sinkhorn_cost(self, r, s, wr, ws, eps=0.05, iters=50):
        """
        r,s: (Sr,D) and (Ss,D) L2-normalized
        wr,ws: (Sr,), (Ss,) masses sum to 1
        returns scalar cost ~ <T, C> with C = 1 - r @ s^T
        """
        # cos in [-1,1]; (cos-1)/eps <= 0 -> K is well-conditioned for small eps
        K = torch.exp((r @ s.t() - 1.0) / eps)
        u = torch.ones_like(wr) / wr.numel()
        v = torch.ones_like(ws) / ws.numel()

        for _ in range(iters):
            u = wr / (K @ v).clamp_min(1e-12)
            v = ws / (K.t() @ u).clamp_min(1e-12)

        T = torch.diag(u) @ K @ torch.diag(v)            # (Sr,Ss)
        C = 1.0 - (r @ s.t())
        return (T * C).sum()                              # scalar

    # =========================
    # Forward
    # =========================
    def forward(self, real_pixel_values, synthetic_pixel_values, labels=None, attention_mask=None):
        # --- Encode real + synthetic (multi-scale aware) ---
        real_feats_list  = self._encode_feats_list(real_pixel_values)
        synth_feats_list = self._encode_feats_list(synthetic_pixel_values)

        if self.multi_scale:
            real_prev, real_last   = real_feats_list
            synth_prev, synth_last = synth_feats_list
        else:
            real_last   = real_feats_list[-1]
            synth_last  = synth_feats_list[-1]

        real_proj_last, (H_l, W_l) = self._project_norm(real_last, use_prev=False)
        synth_proj_last, _          = self._project_norm(synth_last, use_prev=False)
        S_last = H_l * W_l

        # *** Ensure encoder outputs match decoder dtype ***
        real_proj_last = real_proj_last.to(self.dec_dtype)

        # CE via decoder
        enc_out = BaseModelOutput(last_hidden_state=real_proj_last)
        dec_out = self.language_decoder(
            encoder_outputs=enc_out,
            labels=labels,
            decoder_attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True
        )
        ce = dec_out.loss

        # patch weights from cross-attn (no grad)
        with torch.no_grad():
            w_last = self._attention_to_patch_weights(dec_out.cross_attentions, attention_mask, S_last)  # (B,S_last)

        # --- PAL on last scale (fp32) ---
        with torch.amp.autocast('cuda', enabled=False):
            r = real_proj_last.float()
            s = synth_proj_last.float()
            w = w_last.float().unsqueeze(-1)
            r_pool = (r * w).sum(dim=1)
            s_pool = (s * w).sum(dim=1)
            sim = F.cosine_similarity(r_pool, s_pool, dim=-1, eps=1e-8).clamp(-1.0, 1.0)
            pal_last = (1.0 - sim).mean()
            total_pal = pal_last

            # multi-scale PAL: align the penultimate stage too
            if self.multi_scale:
                real_proj_prev, (H_p, W_p) = self._project_norm(real_prev, use_prev=True)
                synth_proj_prev, _ = self._project_norm(synth_prev, use_prev=True)

                # upsample weights from (H_l, W_l) -> (H_p, W_p)
                w_grid = w_last.view(w_last.size(0), 1, H_l, W_l)
                w_prev = F.interpolate(
                    w_grid, size=(H_p, W_p), mode='bilinear', align_corners=False
                ).flatten(2).squeeze(1)
                w_prev = w_prev / w_prev.sum(dim=1, keepdim=True).clamp_min(1e-8)

                rp = real_proj_prev.float()
                sp = synth_proj_prev.float()
                wp = w_prev.float().unsqueeze(-1)
                rp_pool = (rp * wp).sum(dim=1)
                sp_pool = (sp * wp).sum(dim=1)
                sim_p = F.cosine_similarity(rp_pool, sp_pool, dim=-1, eps=1e-8).clamp(-1.0, 1.0)
                pal_prev = (1.0 - sim_p).mean()
                total_pal = 0.5 * (pal_last + pal_prev)

            # InfoNCE (optional)
            nce = r_pool.new_tensor(0.0)
            if self.use_info_nce and r_pool.size(0) >= 2:
                r_n = _l2n(r_pool)
                s_n = _l2n(s_pool)
                feats = torch.cat([r_n, s_n], dim=0)               # (2B,D)
                logits = (feats @ feats.t()) / max(1e-6, self.info_nce_temp)
                logits.fill_diagonal_(-1e9)
                idx = torch.arange(feats.size(0), device=feats.device)
                pos = idx ^ (feats.size(0)//2)                     # i <-> i^B
                nce = -F.log_softmax(logits, dim=1)[idx, pos].mean()

            # OT/Sinkhorn (optional, per-sample loop)
            ot_cost = r_pool.new_tensor(0.0)
            if self.use_ot:
                B, S = r.size(0), r.size(1)
                if 0.0 < self.ot_topk_ratio < 1.0:
                    k = max(1, min(int(self.ot_topk_ratio * S), S))
                    wv, wi = torch.topk(w_last, k, dim=1)
                else:
                    k = S
                    wi = torch.arange(S, device=r.device).unsqueeze(0).expand(B, -1)
                    wv = w_last
                for b in range(B):
                    ridx = wi[b]
                    sidx = wi[b]  # symmetric topk on synth too (proxy)
                    rnorm = _l2n(r[b, ridx, :], dim=-1)
                    snorm = _l2n(s[b, sidx, :], dim=-1)
                    wr = (wv[b] / wv[b].sum()).detach()
                    ws = (wv[b] / wv[b].sum()).detach()
                    ot_cost = ot_cost + self._sinkhorn_cost(
                        rnorm, snorm, wr, ws, eps=self.ot_reg, iters=self.ot_iters
                    )
                ot_cost = ot_cost / B

        total = ce + self.patch_alignment_weight * total_pal
        if self.use_info_nce:
            total = total + (self.info_nce_coef * self.patch_alignment_weight) * nce
        if self.use_ot:
            total = total + (self.ot_coef * self.patch_alignment_weight) * ot_cost

        return {
            "loss": total,
            "cross_entropy_loss": ce,
            "patch_alignment_loss": total_pal,
            "info_nce_loss": (nce if self.use_info_nce else None),
            "ot_loss": (ot_cost if self.use_ot else None),
            "logits": dec_out.logits
        }

    # =========================
    # Generation
    # =========================
    @torch.no_grad()
    def generate(
        self,
        real_pixel_values,
        tokenizer,
        max_new_tokens: int = 128,
        num_beams: int = 6,
        num_return_sequences: int = None,  # default None → auto set to num_beams
        length_penalty: float = 1.3,
        no_repeat_ngram_size: int = 3,
        min_new_tokens: int = 16,
        repetition_penalty: float = 1.02
    ):
        self.eval()
        # encode only last scale for decoding
        last = self._encode_feats_list(real_pixel_values)[-1]
        enc_last, _ = self._project_norm(last, use_prev=False)
    
        # match decoder dtype
        dec_dtype = self.dec_dtype
        enc_last = enc_last.to(dtype=dec_dtype)
    
        encoder_outputs = BaseModelOutput(last_hidden_state=enc_last)
    
        bos_id = tokenizer.lang_code_to_id.get("bn_IN")
        if bos_id is None:
            raise ValueError("bn_IN not found in tokenizer.lang_code_to_id")
        eos_id = tokenizer.eos_token_id
        pad_id = tokenizer.pad_token_id
    
        # if num_return_sequences not specified, generate as many as beams
        if num_return_sequences is None:
            num_return_sequences = num_beams
    
        prev_cache = getattr(self.language_decoder.config, "use_cache", True)
        self.language_decoder.config.use_cache = True
        try:
            out = self.language_decoder.generate(
                encoder_outputs=encoder_outputs,
                decoder_start_token_id=bos_id,
                forced_bos_token_id=bos_id,
                eos_token_id=eos_id,
                pad_token_id=pad_id,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                length_penalty=max(1.0, float(length_penalty)),
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                early_stopping=True,
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                return_dict_in_generate=True
            )
        finally:
            self.language_decoder.config.use_cache = prev_cache
    
        return tokenizer.batch_decode(
            out.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )


In [ ]:
# =============================
# Incremental training (7 epochs)
# with CE + PA + InfoNCE + OT
# and proper resume (model/opt/sched/scaler)
# =============================
import os, json
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
import timm
from transformers import AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput


# --- Enable-at-epoch switches (keep yours) ---
NCE_ENABLE_AT_EPOCH = 0
OT_ENABLE_AT_EPOCH  = 0

# -----------------------------
# Checkpoint helpers
# -----------------------------
def save_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = {
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
    }
    torch.save(ckpt, path)
    print(f"✅ Saved checkpoint to {path}")

def load_checkpoint_if_any(path, device, model, optimizer, scheduler, scaler):
    if not os.path.exists(path):
        print("ℹ️ No previous checkpoint found, training from scratch for this batch.")
        return
    ckpt = torch.load(path, map_location=device)
    # Support both "full" checkpoint and legacy "weights only"
    if "model_state_dict" in ckpt:
        missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
        print(f"Model loaded. Missing keys: {missing} | Unexpected: {unexpected}")
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            scaler.load_state_dict(ckpt["scaler_state_dict"])
            print("Optimizer/Scheduler/Scaler states loaded.")
        except Exception as e:
            print(f"⚠️ Could not load optimizer/scheduler/scaler states: {e} (will reinitialize them)")
    else:
        missing, unexpected = model.load_state_dict(ckpt, strict=False)
        print(f"Legacy weights loaded. Missing keys: {missing} | Unexpected: {unexpected}")

# -----------------------------
# Progressive unfreezing
# (apply each epoch, before training loop)
# -----------------------------
def unfreeze_layers(model, epoch, total_epochs):
    # identical logic you used before
    if epoch >= total_epochs // 3:
        for p in model.language_decoder.model.encoder.layers[0].parameters():
            p.requires_grad = True
    if epoch >= total_epochs // 2:
        for p in model.language_decoder.model.encoder.layers[1].parameters():
            p.requires_grad = True
    if epoch >= 2 * total_epochs // 3:
        for p in model.language_decoder.model.encoder.parameters():
            p.requires_grad = True

# -----------------------------
# Train for exactly 7 epochs
# -----------------------------
def train_for_seven_epochs(
    model, dataloader, optimizer, scheduler, scaler,
    device, grad_accum_steps=GRAD_ACCUM_STEPS, num_epochs=7
):
    # IMPORTANT: create optimizer with ALL params so later unfreezing works
    # (we already created optimizer before calling this with model.parameters())

    for epoch in range(num_epochs):
        model.train()
        total_epoch_loss = 0.0
        total_ce_loss    = 0.0
        total_pa_loss    = 0.0
        total_nce_loss   = 0.0
        total_ot_loss    = 0.0
        num_batches_processed = 0

        # Unfreeze progressively
        unfreeze_layers(model, epoch, num_epochs)

        optimizer.zero_grad(set_to_none=True)

        for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")):
            if batch is None:
                print(f"Warning: Skipping empty batch {batch_idx+1} in Epoch {epoch+1}.")
                continue

            real_pixel_values      = batch["real_pixel_values"].to(device, non_blocking=True)
            synthetic_pixel_values = batch["synthetic_pixel_values"].to(device, non_blocking=True)
            labels                 = batch["labels"].to(device, non_blocking=True)
            decoder_attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            use_nce = USE_INFO_NCE and (epoch >= NCE_ENABLE_AT_EPOCH)
            use_ot  = USE_OT       and (epoch >= OT_ENABLE_AT_EPOCH)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                outputs = model(
                    real_pixel_values=real_pixel_values,
                    synthetic_pixel_values=synthetic_pixel_values,
                    labels=labels,
                    attention_mask=decoder_attention_mask
                )

                # Base CE loss (your model should return this in outputs)
                ce_loss = outputs.get("cross_entropy_loss", outputs.get("loss", 0.0))
                pa_loss = outputs.get("patch_alignment_loss", 0.0)
                nce_loss = outputs.get("info_nce_loss", 0.0) if use_nce else 0.0
                ot_loss  = outputs.get("ot_loss", 0.0)       if use_ot  else 0.0

                # Sum of losses (respect your coefficients)
                total_loss = ce_loss + pa_loss
                if use_nce and isinstance(nce_loss, (int, float)) is False:
                    total_loss = total_loss + INFO_NCE_CFG["coef"] * nce_loss
                elif use_nce:
                    total_loss = total_loss + INFO_NCE_CFG["coef"] * nce_loss

                if use_ot and isinstance(ot_loss, (int, float)) is False:
                    total_loss = total_loss + OT_CFG["coef"] * ot_loss
                elif use_ot:
                    total_loss = total_loss + OT_CFG["coef"] * ot_loss

                # Gradient accumulation
                step_loss = total_loss / grad_accum_steps

            # Backward
            if scaler.is_enabled():
                scaler.scale(step_loss).backward()
            else:
                step_loss.backward()

            # Optimizer step
            if (batch_idx + 1) % grad_accum_steps == 0:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    # clip only current trainable params
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    optimizer.step()

                optimizer.zero_grad(set_to_none=True)

            # ---- Accounting (safe to float()) ----
            def _to_float(x):
                try:
                    return float(x)
                except Exception:
                    try:
                        return x.item()
                    except Exception:
                        return 0.0

            total_epoch_loss += _to_float(step_loss.detach())  # per-step (already / grad_accum)
            total_ce_loss    += _to_float(ce_loss)
            total_pa_loss    += _to_float(pa_loss)
            total_nce_loss   += _to_float(nce_loss)
            total_ot_loss    += _to_float(ot_loss)
            num_batches_processed += 1

            # free
            del real_pixel_values, synthetic_pixel_values, labels, decoder_attention_mask, outputs, step_loss
            torch.cuda.empty_cache()

        # Your original placement: step scheduler once per epoch
        scheduler.step()

        # Report
        if num_batches_processed > 0:
            avg_loss   = total_epoch_loss / num_batches_processed
            avg_ce     = total_ce_loss    / num_batches_processed
            avg_pa     = total_pa_loss    / num_batches_processed
            avg_nce    = total_nce_loss   / num_batches_processed
            avg_ot     = total_ot_loss    / num_batches_processed

            print(f"Epoch {epoch+1}: "
                  f"Avg Total Loss = {avg_loss:.4f}, "
                  f"Avg CE Loss = {avg_ce:.4f}, "
                  f"Avg PA Loss = {avg_pa:.4f}, "
                  f"Avg InfoNCE Loss = {avg_nce:.4f}, "
                  f"Avg OT Loss = {avg_ot:.4f}")
        else:
            print(f"Epoch {epoch+1}: No batches processed (check data or collate_fn).")

        torch.cuda.empty_cache()

PAL_CFG = {
    "attn_temp": 1.0,
    "topk_ratio": 0.50,
    "use_last_k_cross_layers": 2,
    "pool_factor": 2,
    "multi_scale": True
}

USE_INFO_NCE = False
INFO_NCE_CFG = {
    "temp": 0.07,
    "coef": 0.3
}

USE_OT = False
OT_CFG = {
    "reg": 0.05,
    "iters": 30,
    "topk_ratio": 0.10,
    "coef": 0.5
}


if __name__ == "__main__":
    print("--- Starting Data Preprocessing ---")

    # Load COCO annotations
    with open(COCO_ANNOTATIONS_PATH, 'r') as f:
        coco_data = json.load(f)
    annotations = coco_data['annotations']
    df_mscoco = pd.DataFrame(annotations)[['id', 'image_id', 'caption']]
    df_mscoco.rename(columns={'id': 'caption_id', 'caption': 'caption_en'}, inplace=True)

    results_data_list = []

    print(f"Scanning {DATA_DIR} for JSON files...")
    json_files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".json"))

    for j in tqdm(json_files, desc="Processing JSON files"):
        try:
            with open(os.path.join(DATA_DIR, j), encoding="utf-8") as f:
                meta = json.load(f)

            full_caption = meta["caption_bn"]
            caption_en, caption_bn = extract_captions(full_caption)
            caption_variants = handle_full_stop_variation(caption_en)

            matched_rows = None
            for variant in caption_variants:
                matched_rows = df_mscoco[df_mscoco['caption_en'] == variant]
                if not matched_rows.empty:
                    break

            if matched_rows is None or matched_rows.empty:
                continue

            image_id = matched_rows['image_id'].values[0]
            image_id_str = str(image_id).zfill(12)

            real_image_path = os.path.join(REAL_IMAGE_BASE_DIR, f"COCO_train2014_{image_id_str}.jpg")
            generated_image_path = os.path.join(DATA_DIR, meta["filename"])

            if os.path.exists(real_image_path) and os.path.exists(generated_image_path):
                results_data_list.append({
                    "real_image_path": real_image_path,
                    "generated_image_path": generated_image_path,
                    "caption_bn": caption_bn,
                    "caption_en": caption_en,
                    "valid": True
                })
        except Exception as e:
            print(f"❌ Error processing {j}: {e}")

    df_results = pd.DataFrame(results_data_list)
    print(f"--- Data Preprocessing Complete. Found {len(df_results)} valid pairs. ---")
    if df_results.empty:
        print("No valid data pairs found. Please check your data paths and structure.")
        raise SystemExit
    print(df_results.head())

    # -----------------------------
    # Initialize model and tokenizer
    # -----------------------------
    tokenizer = MBart50TokenizerFast.from_pretrained(
        MBART_MODEL_NAME, src_lang="bn_IN", tgt_lang="bn_IN"
    )
    bn_in_token_id = tokenizer.lang_code_to_id.get("bn_IN")

    # Preflight (use your existing helpers)
    DEBUG_PREFLIGHT = False
    transform_for_preflight = debug_transform if DEBUG_PREFLIGHT else train_transform_safe
    keep_idx = preflight_filter(
        df_results,
        tokenizer,
        image_transform=transform_for_preflight,
        max_length=MAX_CAPTION_LENGTH,
        use_debug=DEBUG_PREFLIGHT
    )
    df_results = df_results.loc[keep_idx].reset_index(drop=True)
    print(f"After preflight: {len(df_results)} usable pairs (from {len(keep_idx)}).")

    # -----------------------------
    # Split last 4000 pairs for test
    # -----------------------------
    if len(df_results) > 1000:
        df_train = df_results.iloc[:-1000].reset_index(drop=True)
        df_test = df_results.iloc[-1000:].reset_index(drop=True)
    else:
        df_train = df_results.copy()
        df_test = pd.DataFrame(columns=df_results.columns)

    dataset = BengaliCaptionDataset(df_train, train_transform_safe, tokenizer, max_length=MAX_CAPTION_LENGTH)
    num_workers = 0
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        collate_fn=collate_fn,
        num_workers=num_workers,
        persistent_workers=(num_workers > 0)
    )
    
    # Build model with all options
    model = MaxVitMbartCaptioningModel(
        MAXVIT_MODEL_NAME,
        MBART_MODEL_NAME,
        bn_in_token_id=bn_in_token_id,
        patch_alignment_weight=INITIAL_PATCH_ALIGNMENT_WEIGHT,
        attn_temp=PAL_CFG["attn_temp"],
        topk_ratio=PAL_CFG["topk_ratio"],
        use_last_k_cross_layers=PAL_CFG["use_last_k_cross_layers"],
        pool_factor=PAL_CFG["pool_factor"],
        multi_scale=PAL_CFG["multi_scale"],
        use_info_nce=False,
        info_nce_temp=INFO_NCE_CFG["temp"],
        info_nce_coef=INFO_NCE_CFG["coef"],
        use_ot=False,
        ot_reg=OT_CFG["reg"],
        ot_iters=OT_CFG["iters"],
        ot_topk_ratio=OT_CFG["topk_ratio"],
        ot_coef=OT_CFG["coef"]
    ).to(DEVICE)


    for p in model.language_decoder.parameters():
        p.requires_grad = False
    
    # 3) Create optimizer **with ALL params** so later unfreezing gets updated
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    
    # 4) Scheduler (same config you used)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LEARNING_RATE,
        epochs=NUM_EPOCHS,                 # you set this to 7
        steps_per_epoch=len(dataloader),
        pct_start=0.1,
        anneal_strategy='linear',
        cycle_momentum=False
    )
    
    # 5) Grad scaler
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    # 6) Resume if checkpoint exists (loads model/opt/sched/scaler)
    if os.path.exists(MODEL_CHECKPOINT_PATH_IN):
        load_checkpoint_if_any(MODEL_CHECKPOINT_PATH_IN, DEVICE, model, optimizer, scheduler, scaler)
    else:
        print("ℹ️ No MODEL_CHECKPOINT_PATH found; if MODEL_INPUT_PATH exists, load raw weights for warm start.")
        if os.path.exists(MODEL_INPUT_PATH):
            state = torch.load(MODEL_INPUT_PATH, map_location=DEVICE)
            missing, unexpected = model.load_state_dict(state, strict=False)
            print(f"Warm-start from MODEL_INPUT_PATH. Missing: {missing} | Unexpected: {unexpected}")
    
    # 7) Train exactly 7 epochs on this batch
    train_for_seven_epochs(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        num_epochs=NUM_EPOCHS  # set to 7 in your config
    )
    
    # 8) Save single checkpoint (no epoch number stored)
    save_checkpoint(MODEL_CHECKPOINT_PATH, model, optimizer, scheduler, scaler)
    
    print("✅ Incremental run complete. Ready for the next 15–20k batch.")


In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/salaniz/pycocoevalcap.git

# Step 2: Install it
!cd pycocoevalcap && python setup.py install

# Step 3: Add it to sys.path so Python can find it immediately
import sys
sys.path.append("/kaggle/working/pycocoevalcap")

!pip install -q bert-score

In [ ]:
# -*- coding: utf-8 -*-
import os, time
import torch
from torch import nn
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
import torchvision.transforms as transforms
import datetime

from transformers import MBart50TokenizerFast, pipeline
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# Pycocoevalcap
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

# Optional: BERTScore
try:
    from bert_score import score as bert_score
    USE_BERTSCORE = True
except:
    USE_BERTSCORE = False

# ------------------------
# Config / paths
# ------------------------
FLICKR30K_DIR = "/kaggle/input/flickr30k"
CAPTIONS_FILE = os.path.join(FLICKR30K_DIR, "captions.txt")
FLICKR_IMAGES_DIR = os.path.join(FLICKR30K_DIR, "flickr30k_images")

MODEL_SAVE_PATH = MODEL_CHECKPOINT_PATH
OUTPUT_RESULTS_CSV = "/kaggle/working/flickr30k_bengali_caption_results.csv"

NUM_TEST_IMAGES = 1000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAXVIT_MODEL_NAME = "maxvit_base_tf_224.in1k"
MBART_MODEL_NAME  = "facebook/mbart-large-50-many-to-many-mmt"

BATCH_SIZE = 8
TRANSLATION_BATCH = 32
TOP_N = 5

import warnings
warnings.filterwarnings("ignore")

# ------------------------
# Utils
# ------------------------
def safe_load_image(img_path: str):
    try:
        img = Image.open(img_path).convert('RGB')
        return img
    except Exception as e:
        print(f"ERROR loading image {img_path}: {e}")
        return None

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

# NLTK punkt
try:
    nltk.data.find('tokenizers/punkt')
except:
    nltk.download('punkt')

# ------------------------
# Load tokenizer + model
# ------------------------
tokenizer = MBart50TokenizerFast.from_pretrained(MBART_MODEL_NAME)
tokenizer.src_lang = "bn_IN"
tokenizer.tgt_lang = "bn_IN"
bn_in_token_id = tokenizer.lang_code_to_id.get("bn_IN")

# --- Load your MaxVitMbartCaptioningModel class here ---
# from your_model_file import MaxVitMbartCaptioningModel
model = MaxVitMbartCaptioningModel(
    MAXVIT_MODEL_NAME, MBART_MODEL_NAME, bn_in_token_id=bn_in_token_id
).to(DEVICE)

def load_weights_for_eval(model: nn.Module, ckpt_path: str, device: str = "cpu"):
    """
    Robustly load weights for evaluation from either:
      - a full checkpoint dict (expects 'model_state_dict'), or
      - a raw state_dict (plain key->tensor mapping).
    Strips 'module.' prefixes if present and reports key mismatches.
    """
    ckpt = torch.load(ckpt_path, map_location=device)

    # pick the state_dict
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:  # some trainers use this key
        state = ckpt["state_dict"]
    else:
        state = ckpt  # assume it's already a raw state_dict

    # strip DataParallel/DDP prefix if needed
    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"[load_weights_for_eval] Missing keys: {missing}")
        print(f"[load_weights_for_eval] Unexpected keys: {unexpected}")

    model.eval()
    return missing, unexpected


load_weights_for_eval(model, MODEL_SAVE_PATH, device="cpu")
model = model.to(DEVICE)
model.eval()

# ------------------------
# Translation
# ------------------------
translator_pipe = pipeline("translation", model="shhossain/opus-mt-en-to-bn", device=-1)

def translate_batch(batch):
    res = []
    for s in batch:
        try:
            res.append(translator_pipe(s)[0]['translation_text'])
        except:
            res.append("")
    return res

# ------------------------
# Read Flickr30k captions
# ------------------------
image_captions = {}
with open(CAPTIONS_FILE, 'r') as f:
    for line in f:
        parts = line.strip().split(',', 2)
        if len(parts) == 3:
            filename, _, caption = parts
            image_captions.setdefault(filename, []).append(caption)

existing_images = {f for f in os.listdir(FLICKR_IMAGES_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))}
image_captions_filtered = {fn: caps for fn, caps in image_captions.items() if fn in existing_images}

items = list(image_captions_filtered.items())
if NUM_TEST_IMAGES:
    items = items[:NUM_TEST_IMAGES]

# ------------------------
# Scorers
# ------------------------
chencherry = SmoothingFunction()
meteor_scorer = Meteor()
rouge_scorer = Rouge()
cider_scorer = Cider()
spice_scorer = Spice()

# ------------------------
# Tokenizer helper
# ------------------------
def tokenize_bn_model(sentences, tokenizer):
    tokenized = []
    for s in sentences:
        # print(f"Tokenizing sentence: {s}")  # Debugging: Print the sentence being tokenized
        if s.strip():
            tokens = tokenizer.convert_ids_to_tokens(tokenizer(s, return_tensors="pt")["input_ids"][0])
            tokenized.append(tokens)
        else:
            print(f"Skipping empty or invalid sentence: {s}")  # Debugging: Show if empty sentence
    return tokenized

# ------------------------
# Main evaluation
# ------------------------
results_list = []
start_time = time.time()

for img_filename, eng_refs in tqdm(items, desc="Processing Images"):
    image_path = os.path.join(FLICKR_IMAGES_DIR, img_filename)
    
    # Translate English references to Bengali
    bn_refs = []
    for i in range(0, len(eng_refs), TRANSLATION_BATCH):
        batch = eng_refs[i:i+TRANSLATION_BATCH]
        bn_refs.extend(translate_batch(batch))
    
    # Check if there are references available
    if len(bn_refs) == 0:
        print(f"No Bengali references found for {img_filename}. Skipping image.")
        continue

    img = safe_load_image(image_path)
    if img is None:
        continue

    with torch.no_grad():
        with torch.amp.autocast(device_type=DEVICE, dtype=torch.float16 if DEVICE=="cuda" else torch.float32):
            img_t = eval_transform(img).unsqueeze(0).to(DEVICE)
            gen_list = model.generate(
                real_pixel_values=img_t,
                tokenizer=tokenizer,
                max_new_tokens=128,
                num_beams=TOP_N,
                length_penalty=1.3,
                no_repeat_ngram_size=3,
                min_new_tokens=16,
                repetition_penalty=1.02
            )

    # Check that the generated captions are not empty
    if len(gen_list) == 0:
        print(f"No generated captions for {img_filename}. Skipping image.")
        continue

    # Tokenize the references and generated captions
    tokenized_refs = tokenize_bn_model(bn_refs, tokenizer)
    tokenized_preds = [tokenize_bn_model([gen], tokenizer)[0] for gen in gen_list]

    best_scores = {
        "BLEU-1": 0, "BLEU-2": 0, "BLEU-3": 0, "BLEU-4": 0,
        "METEOR": 0, "CIDEr": 0, "ROUGE-L": 0, "SPICE": 0
    }
    if USE_BERTSCORE:
        best_scores["BERTScore-F1"] = 0.0

    best_caption = gen_list[0]

    for candidate, tokenized_pred in zip(gen_list, tokenized_preds):
        # --- BLEU 1-4 with smoothing ---
        bleu_scores = []
        for n in range(1, 5):
            try:
                bleu_n = corpus_bleu(
                    [tokenized_refs],
                    [tokenized_pred],
                    weights=tuple([1.0/n]*n) + tuple([0]*(4-n)),
                    smoothing_function=chencherry.method1
                ) * 100
            except:
                bleu_n = 0.0
            bleu_scores.append(bleu_n)
        bleu1, bleu2, bleu3, bleu4 = bleu_scores

        try:
            # CIDEr / METEOR / ROUGE-L / SPICE (Ensure proper input format for CIDEr)
            meteor_score = meteor_scorer.compute_score({img_filename: bn_refs}, {img_filename: [candidate]})[0]
            cider_score  = cider_scorer.compute_score({img_filename: bn_refs}, {img_filename: [candidate]})[0]
            rouge_score  = rouge_scorer.compute_score({img_filename: bn_refs}, {img_filename: [candidate]})[0]
            spice_score  = spice_scorer.compute_score({img_filename: bn_refs}, {img_filename: [candidate]})[0]
        except Exception as e:
            print(f"Error calculating scores for {img_filename}, {candidate}: {e}")
            meteor_score = cider_score = rouge_score = spice_score = 0.0

        # --- BERTScore (conditional) ---
        bert_f1 = 0.0
        if USE_BERTSCORE:
            try:
                # Ensure that BERTScore is comparing all references to the current candidate
                P, R, F1 = bert_score([candidate] * len(bn_refs), bn_refs, lang="bn", rescale_with_baseline=False)
                bert_f1 = float(F1.mean())
            except Exception as e:
                print(f"Error calculating BERTScore for {img_filename}: {e}")
                bert_f1 = 0.0

        metric_scores = {
            "BLEU-1": bleu1,
            "BLEU-2": bleu2,
            "BLEU-3": bleu3,
            "BLEU-4": bleu4,
            "METEOR": meteor_score,
            "CIDEr": cider_score,
            "ROUGE-L": rouge_score,
            "SPICE": spice_score,
            "BERTScore-F1": bert_f1
        }

        # Pick best caption by sum of all metrics
        if sum(metric_scores.values()) > sum(best_scores.values()):
            best_scores = metric_scores
            best_caption = candidate


    results_list.append({
        "image_path": img_filename,
        "bengali_reference": "|".join(bn_refs),
        "predicted_caption_bn": best_caption,
        **best_scores
    })

# Save results
df_results = pd.DataFrame(results_list)
df_results.to_csv(OUTPUT_RESULTS_CSV, index=False)
print("Evaluation finished. Results saved at:", OUTPUT_RESULTS_CSV)

# ------------------------
# Print average metrics
# ------------------------
metrics = ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4", "METEOR", "CIDEr", "ROUGE-L", "SPICE"]
if USE_BERTSCORE:
    metrics.append("BERTScore-F1")

for m in metrics:
    if m in df_results.columns:
        avg = df_results[m].mean()
        print(f"{m}: {avg:.4f}")

elapsed = str(datetime.timedelta(seconds=int(time.time() - start_time)))
print(f"\nTotal processing time: {elapsed}")


In [ ]:
import time
import datetime
import torch
import pandas as pd
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from bert_score import score as bert_score

# ------------------------
# Config
# ------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOP_N = 6
USE_BERTSCORE = True  # Flag to enable/disable BERTScore

results_list = []
start_time = time.time()

# ------------------------
# Evaluation
# ------------------------
for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Processing images"):
    img_path = row["real_image_path"]
    reference_bn = [row["caption_bn"]]  # list of Bengali references

    # Load image safely
    img = safe_load_image(img_path)
    if img is None:
        continue

    # Generate top-N captions
    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE=="cuda")):
            img_t = eval_transform(img).unsqueeze(0).to(DEVICE)
            gen_list = model.generate(
                real_pixel_values=img_t,
                tokenizer=tokenizer,
                max_new_tokens=128,
                num_beams=TOP_N,
                length_penalty=1.3,
                no_repeat_ngram_size=3,
                min_new_tokens=16,
                repetition_penalty=1.02
            )

    if len(gen_list) == 0:
        print(f"No generated captions for {img_path}, skipping.")
        continue

    # Tokenize references once
    tokenized_refs_metric = tokenize_bn_model(reference_bn, tokenizer)

    # Initialize best scores
    best_scores = {m: -1 for m in ["BLEU-1","BLEU-2","BLEU-3","BLEU-4",
                                   "METEOR","CIDEr","ROUGE-L","SPICE","BERTScore-F1"]}
    best_caption = gen_list[0]

    for candidate in gen_list:
        tokenized_pred_metric = tokenize_bn_model([candidate], tokenizer)[0]

        # --- BLEU 1-4 ---
        weights_list = [(1,0,0,0), (0.5,0.5,0,0), (0.33,0.33,0.33,0), (0.25,0.25,0.25,0.25)]
        bleu_scores = []
        for w in weights_list:
            try:
                bleu = corpus_bleu(
                    [tokenized_refs_metric],
                    [tokenized_pred_metric],
                    weights=w,
                    smoothing_function=SmoothingFunction().method1
                ) * 100
            except:
                bleu = 0.0
            bleu_scores.append(bleu)
            
        bleu1, bleu2, bleu3, bleu4 = bleu_scores

        # --- METEOR / CIDEr / ROUGE-L / SPICE ---
        try:
            image_id = idx  # unique ID per row
            refs_dict = {image_id: reference_bn}       # list of references
            hyps_dict = {image_id: [candidate]}        # candidate wrapped in list
        
            meteor_score = float(meteor_scorer.compute_score(refs_dict, hyps_dict)[0])
            cider_score  = float(cider_scorer.compute_score(refs_dict, hyps_dict)[0])
            rouge_l_score, _ = rouge_scorer.compute_score(refs_dict, hyps_dict)   # <- tuple unpack
            spice_score  = float(spice_scorer.compute_score(refs_dict, hyps_dict)[0])
        except Exception as e:
            print(f"Metric error for {img_path}, {candidate}: {e}")
            meteor_score = cider_score = rouge_l_score = spice_score = 0.0


        # --- BERTScore ---
        bert_f1 = 0.0
        if USE_BERTSCORE:
            try:
                P, R, F1 = bert_score([candidate]*len(reference_bn), reference_bn, lang="bn", rescale_with_baseline=False)
                bert_f1 = float(F1.mean())
            except Exception as e:
                print(f"BERTScore error for {img_path}: {e}")
                bert_f1 = 0.0

        metric_scores = {
            "BLEU-1": bleu1,
            "BLEU-2": bleu2,
            "BLEU-3": bleu3,
            "BLEU-4": bleu4,
            "METEOR": meteor_score,
            "CIDEr": cider_score,
            "ROUGE-L": rouge_l_score,
            "SPICE": spice_score,
            "BERTScore-F1": bert_f1
        }

        # Pick best caption by sum of all metrics
        if sum(metric_scores.values()) > sum(best_scores.values()):
            best_scores = metric_scores
            best_caption = candidate

    results_list.append({
        "image_path": img_path,
        "bengali_reference": "|".join(reference_bn),
        "predicted_caption_bn": best_caption,
        **best_scores
    })

# ------------------------
# Save results
# ------------------------
out_df = pd.DataFrame(results_list)
OUTPUT_RESULTS_CSV = "/kaggle/working/df_test_generated_bestofN_scores.csv"
out_df.to_csv(OUTPUT_RESULTS_CSV, index=False)
print(f"Results saved at: {OUTPUT_RESULTS_CSV}")

# ------------------------
# Print average metrics
# ------------------------
metrics = ["BLEU-1","BLEU-2","BLEU-3","BLEU-4","METEOR","CIDEr","ROUGE-L","SPICE"]
if USE_BERTSCORE:
    metrics.append("BERTScore-F1")

for m in metrics:
    if m in out_df.columns:
        print(f"{m} Avg: {out_df[m].mean():.4f}")

elapsed = str(datetime.timedelta(seconds=int(time.time() - start_time)))
print(f"\nTotal processing time: {elapsed}")
